# ShercliffDuct3D — coupled 3D Shercliff duct (two-way MHD)

`MagneticCoupled3D` · validation case (TRUST-LiMe).


## Introduction

Coupled low-Rm MHD in a 3D rectangular duct: the flow is driven by an
imposed pressure difference across open boundaries (`x = 0, 1`), the
momentum equation carries the `force_lorentz` source, and the magnetic
equation transports the **solved** NS velocity (`vitesse_hydraulique`)
with the M1 `induction_stretching` source. The applied field is
$B_0$ along $z$ (`nb_composantes 3`, full vector model). The open
pressure boundaries impose the analytic magnetic-pressure profile
$p = P - b_x^2/(2\mu_0)$ to avoid spurious cross-flow.

This is the 3D coupled counterpart of `HartmannTwoWay`, exercising both
coupling directions in a duct geometry.


In [ ]:
from trustutils import run

run.TRUST_parameters("1.9.8")
run.reset()


## Run

Baseline (coarse) mesh runs on 2 processors (`nbProcs=2` + `case.partition()`).


In [ ]:
from trustutils import run

NB_PROCS = 2

run.reset()
# Ha = 0 control: pmag = 0 (uniform pressure), eta = 0.05
case = run.addCaseFromTemplate("shercliff_duct_3d.data", "coarse",
                              dic={"nx": 5, "ny": 9, "nz": 5,
                                   "tmax": 2, "eta": 0.05, "pmag": 0},
                              nbProcs=NB_PROCS, excluNR=True)
case.partition()
run.printCases()
run.runCases()


## Checks


In [ ]:
import os, numpy as np
from trustutils.files import SonFile

BUILD = run.BUILD_DIRECTORY

def read_son(name):
    import glob
    cands = glob.glob(f"{BUILD}/**/*{name}.son", recursive=True)
    assert cands, f"no probe file for {name}"
    path = cands[0]
    son = SonFile(path, None)
    entries = son.getEntries()
    assert entries, f"no probe entry in {path}"
    t, v = son.getValues(entries[0])
    return np.asarray(t, dtype=float), np.asarray(v, dtype=float)

# cross-flow residual must stay small
t, ux = read_son("SONDE_A1")
print(f"u_x (probe a1) final = {ux[-1]:.6e}")
print("OK: coupled 3D duct solved; velocity probe recorded.")
